In [1]:
import copy
from pathlib import Path
import yaml
import torch
import torch.onnx
from torchao.quantization.prototype.qat import Int8DynActInt4WeightQATQuantizer

Skipping import of cpp extensions due to incompatible torch version 2.10.0+cu130 for torchao version 0.15.0             Please see https://github.com/pytorch/ao/issues/2919 for more info


In [2]:
# from rtal.models.mlp_no_residual import MLP
from mlp_for_quantization import MLP

In [3]:
ROOT = Path('/home/yhuang2/PROJs/RealTimeAlignment/train/mlp_better-target_qat/')
CHECKPOINT_PATH = ROOT/'checkpoints'/'ckpt_last.pth'   # Path to your saved .pth file
ONNX_OUTPUT     = "model_fp32.onnx"  # Destination ONNX file
DEVICE = 'cpu'

with open('config.yaml') as handle:
    config = yaml.safe_load(handle)

model = MLP(**config['model'])

In [16]:
ckpt = torch.load(CHECKPOINT_PATH, map_location=DEVICE, weights_only=False)
model.load_state_dict(ckpt['model'], strict=True)
model.to(DEVICE)
print(ckpt['model'])

OrderedDict({'embed.1.weight': tensor([[-2.5938e-01,  9.1407e-02,  2.4512e-01, -9.0607e-02,  8.4142e-02,
          6.4952e-02],
        [-1.8338e-01, -1.1252e-01, -2.0030e-01, -3.3635e-02,  6.6101e-02,
          2.2199e-01],
        [ 3.4873e-01, -4.2398e-01, -1.7479e-01, -5.6640e-02, -1.7838e-01,
          1.7196e-01],
        [ 1.6804e-02,  8.1799e-02,  5.6861e-02,  2.5107e-01, -2.0826e-02,
         -2.4828e-01],
        [-2.3193e-01,  3.0014e-01,  2.1643e-01, -1.2422e-01,  2.7446e-02,
          1.0566e-01],
        [ 2.9575e-02, -3.1645e-01, -4.2536e-03,  1.0411e-01,  9.9083e-02,
         -1.4284e-01],
        [-2.1810e-01,  7.2225e-02,  2.4614e-01, -3.9079e-02, -1.7869e-02,
          2.1644e-02],
        [ 3.8786e-02, -5.1928e-02, -2.7544e-01, -1.6372e-01,  8.3652e-02,
         -1.9092e-01],
        [ 9.4282e-02,  8.4035e-02, -1.4058e-01, -9.2520e-02,  4.9808e-02,
          6.4666e-02],
        [ 2.3254e-01, -2.2618e-01,  6.7631e-02,  3.5574e-01, -2.3039e-01,
         -3.1069e-01],

In [19]:
qat_quantizer = Int8DynActInt4WeightQATQuantizer()
base_model = MLP(**config['model'])
converted_model = qat_quantizer.prepare(base_model)
converted_model = qat_quantizer.convert(converted_model)

CONVERTED_CHECKPOINT_PATH = ROOT/'checkpoints'/'ckpt_last_converted.pth'
ckpt = torch.load(CONVERTED_CHECKPOINT_PATH, map_location=DEVICE, weights_only=False)
converted_model.load_state_dict(ckpt['model'], strict=True)
converted_model.to(DEVICE)

# Use your already-converted model
converted_model.eval()

# Create a dummy input matching your model's input shape
dummy_input = torch.randn(INPUT_SHAPE)  # adjust shape

try:
    torch.onnx.export(
        converted_model,
        dummy_input,
        "model_quantized.onnx",
        opset_version=17,
        input_names=["input"],
        output_names=["output"],
        dynamic_axes={"input": {0: "batch_size"}, "output": {0: "batch_size"}},
        verbose=False,
        dynamo=False,
    )
    print("Export succeeded")
except Exception as e:
    print(f"Export failed: {e}")



Export failed: iter->isIntList() INTERNAL ASSERT FAILED at "/pytorch/torch/csrc/autograd/TraceTypeManual.cpp":238, please report a bug to PyTorch. 


/tmp/ipykernel_768448/4141455401.py:18: DeprecationWarning: You are using the legacy TorchScript-based ONNX export. Starting in PyTorch 2.9, the new torch.export-based ONNX exporter has become the default. Learn more about the new export logic: https://docs.pytorch.org/docs/stable/onnx_export.html. For exporting control flow: https://pytorch.org/tutorials/beginner/onnx/export_control_flow_model_to_onnx_tutorial.html
  torch.onnx.export(


In [5]:
# Input shape for your MLP — change to match your data
# Examples:
#   1D vector input of size 128         → (1, 128)
#   2D sequence, 32 tokens × 64 dims    → (1, 32, 64)
INPUT_SHAPE = (1, 50, 6)               # (batch_size, input_features)  <-- EDIT THIS

OPSET_VERSION   = 17
DYNAMIC_BATCH   = True               # Set False to fix batch size = 1
DEVICE          = "cpu"

In [6]:
def load_qat_checkpoint(checkpoint_path: str) -> dict:
    """Load checkpoint — handles state_dict, full model, or nested dict formats."""
    raw = torch.load(checkpoint_path, map_location="cpu", weights_only=False)

    if isinstance(raw, dict):
        # Nested dict: e.g. {'model': state_dict, 'epoch': 10, ...}
        if "model" in raw:
            print("Detected nested checkpoint dict — using key 'model'.")
            return raw["model"]
        # Plain state dict
        print("Detected plain state_dict checkpoint.")
        return raw
    else:
        # Full model object saved with torch.save(model, path)
        print("Detected full model checkpoint.")
        return raw.state_dict()


def build_qat_model(state_dict: dict) -> torch.nn.Module:
    """
    Reconstruct the QAT-prepared model and load saved weights.
    We load into the QAT-prepared model (with fake-quant ops) so that
    the parameter shapes match the checkpoint exactly.
    """
    model = MLP(**config["model"])

    quantizer = Int8DynActInt4WeightQATQuantizer()
    model = quantizer.prepare(model)           # Insert fake-quant ops
    model.load_state_dict(state_dict)
    model.eval()
    print("QAT model loaded and set to eval mode.")
    return model


def strip_quantization_to_fp32(qat_model: torch.nn.Module) -> torch.nn.Module:
    """
    Build a clean fp32 MLP and copy dequantized weights from the QAT model.

    The QAT model's Linear layers still hold fp32 weights — the fake-quant
    ops only simulated quantization numerics.  We just need to copy them
    into a plain model, discarding all fake-quant wrapper state.
    """
    plain_model = MLP(**config["model"])
    plain_model.eval()

    # Map both models' named modules for convenient lookup
    qat_modules   = dict(qat_model.named_modules())
    plain_modules = dict(plain_model.named_modules())

    copied, skipped = 0, 0
    with torch.no_grad():
        for name, plain_module in plain_modules.items():
            if not isinstance(plain_module, torch.nn.Linear):
                continue

            qat_module = qat_modules.get(name)
            if qat_module is None:
                print(f"  [WARN] Layer '{name}' not found in QAT model — skipping.")
                skipped += 1
                continue

            # Copy weight (cast to float32 in case of any residual dtype)
            plain_module.weight.copy_(qat_module.weight.float())

            if plain_module.bias is not None and qat_module.bias is not None:
                plain_module.bias.copy_(qat_module.bias.float())

            copied += 1

    print(f"Weights copied: {copied} linear layers  |  skipped: {skipped}")
    return plain_model


def verify_outputs(qat_model: torch.nn.Module,
                   plain_model: torch.nn.Module,
                   input_shape: tuple,
                   atol: float = 1e-4) -> None:
    """Sanity-check: QAT and fp32 model outputs should be very close."""
    dummy = torch.randn(*input_shape)
    with torch.no_grad():
        out_qat   = qat_model(dummy)
        out_plain = plain_model(dummy)

    max_diff = (out_qat - out_plain).abs().max().item()
    status   = "PASS ✓" if max_diff < atol else "WARN ✗ (larger than expected)"
    print(f"Output verification — max absolute diff: {max_diff:.6f}  [{status}]")


# def export_onnx(model: torch.nn.Module,
#                 input_shape: tuple,
#                 output_path: str,
#                 opset: int,
#                 dynamic_batch: bool) -> None:
#     """Export the fp32 model to ONNX."""
#     dummy_input = torch.randn(*input_shape)

#     # Dynamic axes: allow variable batch size at runtime
#     dynamic_axes = None
#     if dynamic_batch:
#         dynamic_axes = {
#             "input":  {0: "batch_size"},
#             "output": {0: "batch_size"},
#         }

#     torch.onnx.export(
#         model,
#         dummy_input,
#         output_path,
#         opset_version=opset,
#         input_names=["input"],
#         output_names=["output"],
#         dynamic_axes=dynamic_axes,
#         do_constant_folding=True,   # fold constants for smaller/faster graph
#         verbose=False,
#     )
#     print(f"ONNX model saved → {output_path}")
    
def export_onnx(model, input_shape, output_path, opset, dynamic_batch):
    dummy_input = (torch.randn(*input_shape),)  # note: must be a tuple

    # Specify dynamic batch size using torch.export.Dim
    dynamic_shapes = None
    if dynamic_batch:
        batch = torch.export.Dim("batch_size")
        dynamic_shapes = {"x": {0: batch}}  # "x" matches the forward(self, x) arg name

    onnx_program = torch.onnx.export(
        model,
        dummy_input,
        dynamo=True,
        opset_version=opset,
        input_names=["input"],
        output_names=["output"],
        dynamic_shapes=dynamic_shapes,
        optimize=True,      # graph optimization, default True in PyTorch 2.7+
        verify=True,        # runs ONNXRuntime check automatically
    )

    onnx_program.save(output_path)
    print(f"ONNX model saved → {output_path}")


def validate_onnx(onnx_path: str, input_shape: tuple) -> None:
    """
    Optional: load the ONNX file with onnxruntime and run a forward pass
    to confirm the file is valid and executable.
    """
    try:
        import onnxruntime as ort
        import numpy as np

        sess = ort.InferenceSession(onnx_path, providers=["CPUExecutionProvider"])
        dummy_np = np.random.randn(*input_shape).astype(np.float32)
        out = sess.run(None, {"input": dummy_np})
        print(f"ONNXRuntime validation PASS ✓  — output shape: {out[0].shape}")

    except ImportError:
        print("onnxruntime not installed — skipping runtime validation.")
        print("  Install with:  pip install onnxruntime")
    except Exception as e:
        print(f"ONNXRuntime validation FAILED: {e}")

In [10]:
print("=" * 60)
print("  QAT → FP32 ONNX Export Pipeline")
print("=" * 60)

# 1. Load checkpoint
print("\n[1/5] Loading checkpoint...")
state_dict = load_qat_checkpoint(CHECKPOINT_PATH)

# 2. Rebuild QAT model with saved weights
print("\n[2/5] Reconstructing QAT model...")
qat_model = build_qat_model(state_dict)

# 3. Strip quantization → clean fp32 model
print("\n[3/5] Stripping quantization to fp32...")
plain_model = strip_quantization_to_fp32(qat_model)

# 4. Verify outputs match
print("\n[4/5] Verifying outputs...")
verify_outputs(qat_model, plain_model, INPUT_SHAPE)

# 5. Export to ONNX
print("\n[5/5] Exporting to ONNX...")
# export_onnx(model         = plain_model,
#             input_shape   = INPUT_SHAPE,
#             output_path   = ONNX_OUTPUT,
#             opset         = OPSET_VERSION,
#             dynamic_batch = DYNAMIC_BATCH)

#     # # Dynamic axes: allow variable batch size at runtime
#     # dynamic_axes = None
#     # if dynamic_batch:
#     #     dynamic_axes = {
#     #         "input":  {0: "batch_size"},
#     #         "output": {0: "batch_size"},
#     #     }

#     # torch.onnx.export(
#     #     model,
#     #     dummy_input,
#     #     output_path,
#     #     opset_version=opset,
#     #     input_names=["input"],
#     #     output_names=["output"],
#     #     dynamic_axes=dynamic_axes,
#     #     do_constant_folding=True,   # fold constants for smaller/faster graph
#     #     verbose=False,
#     # )

dummy_input = torch.randn(*INPUT_SHAPE)
torch.onnx.export(
    plain_model,
    dummy_input,
    ONNX_OUTPUT,
    opset_version=13,
    do_constant_folding=True,
    input_names=['input'],
    output_names=['output'],
    dynamo=False, 
)

# Bonus: validate with onnxruntime
print("\n[Bonus] Validating with ONNXRuntime...")
validate_onnx(ONNX_OUTPUT, INPUT_SHAPE)

print("\nDone! Share", ONNX_OUTPUT, "with your collaborators.")

  QAT → FP32 ONNX Export Pipeline

[1/5] Loading checkpoint...
Detected nested checkpoint dict — using key 'model'.

[2/5] Reconstructing QAT model...
QAT model loaded and set to eval mode.

[3/5] Stripping quantization to fp32...
Weights copied: 15 linear layers  |  skipped: 0

[4/5] Verifying outputs...
Output verification — max absolute diff: 0.015330  [WARN ✗ (larger than expected)]

[5/5] Exporting to ONNX...

[Bonus] Validating with ONNXRuntime...
onnxruntime not installed — skipping runtime validation.
  Install with:  pip install onnxruntime

Done! Share model_fp32.onnx with your collaborators.


/tmp/ipykernel_768448/403472320.py:50: DeprecationWarning: You are using the legacy TorchScript-based ONNX export. Starting in PyTorch 2.9, the new torch.export-based ONNX exporter has become the default. Learn more about the new export logic: https://docs.pytorch.org/docs/stable/onnx_export.html. For exporting control flow: https://pytorch.org/tutorials/beginner/onnx/export_control_flow_model_to_onnx_tutorial.html
  torch.onnx.export(
